In [ ]:
import base64
import requests
import time
import pandas as pd
import os

# Claude API Key
api_key = ""

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Read CSV
questions_df = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/VLAT Questions.csv")
questions = questions_df.to_dict('records')
responses_data = []


for i, question in enumerate(questions, start=1):
    time.sleep(5)
    try:
        print(f"\nProcessing question {i}:")
        print(question)
        
        # Path to your image
        image_path = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/" + str(question.get('vis', '')) + ".png"
        
        # Question text
        question_text = question.get('question: ', '')
        question_options = question.get('option:', '')
        correct_ans = str(question.get('correct', '')).strip()
        
        print(f"Processing image: {image_path}")
        print(f"Question: {question_text}")
        print(f"Options: {question_options}")
        print(f"Correct answer: {correct_ans}")
        
        # Getting the base64 string
        base64_image = encode_image(image_path)
        
        headers = {
            "Content-Type": "application/json",
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01"
        }
        
        payload = {
            "model": "claude-3-7-sonnet-20250219",
            "max_tokens": 5000,
            "temperature": 0.0,
            "system": "You are an assistant, skilled in reading and interpreting visually represented data.",
            "messages": [
                {
                    "role": "user",
                    "content": 'I am about to show you a graph and ask you a multiple-choice question about that graph. \n\n' +
                    
                               'Task 1: Data Extraction and Table Creation: First, explicitly list ALL numerical values you can identify on both axes, then create a structured table using markdown syntax that includes ALL data points you identified above with appropriate column headers with units. \n \n' +
                    
                                'Task 2: Sort the data: Sort the data in descending order by the numerical values. \n \n' +

                                'Task 3: Data Verification and Error Handling: Double-check if your table matches ALL elements in the graph by comparing each value in your table with the graph and updating your table with correct values, verify the sorting is correct, and before proceeding, confirm all corrections have been made and use ONLY the corrected data for analysis. \n \n' +

                                'Task 4: Question Analysis: Using ONLY the verified data in your table, compare EACH value individually with the reference value, for "less than" comparisons mark ALL values that are even slightly below the reference, for "greater than" comparisons mark ALL values that are even slightly above the reference, and show each comparison on a new line. \n \n' +

                                'Provide your reasoning with specific references to table values. \n\n' +
                    
                                'End with: "Correct Answer: ". Just write the value, nothing else. Do not write anything after this. \n\n' +
                    
                                'Let\'s solve this step by step.' 



                 
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": question_text + " " + question_options
                        },
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/png",
                                "data": base64_image
                            }
                        }
                    ]
                }
            ]
        }
        
        # Add retry logic for overloaded errors
        max_retries = 3
        retry_delay = 20  # seconds
        
        for retry in range(max_retries):
            try:
                time_start = time.perf_counter()
                response_raw = requests.post(
                    "https://api.anthropic.com/v1/messages",
                    headers=headers,
                    json=payload
                )
                time_end = time.perf_counter()
                
                response = response_raw.json()
                
                # If we get an overloaded error, wait and retry
                if 'error' in response and response['error'].get('type') == 'overloaded_error':
                    if retry < max_retries - 1:  # Don't sleep on the last retry
                        print(f"\nAPI overloaded, waiting {retry_delay} seconds before retry {retry + 1}/{max_retries}")
                        time.sleep(retry_delay)
                        continue
                
                break  # If we get here, we either got a good response or a different error
                
            except Exception as e:
                print(f"Error during API call (attempt {retry + 1}/{max_retries}):", str(e))
                if retry < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                raise  # Re-raise the last exception if we've exhausted all retries
        print("\nAPI Response:", response)  # Debug print
        
        if 'error' in response:
            gpt_answer = f"{response['error']['type']}: {response['error']['message']}"
            is_correct = "N/A"
        else:
            # Extract the answer after "Correct Answer: "
            full_response = response["content"][0]["text"].strip()
            if "Correct Answer: " in full_response:
                gpt_answer = full_response.split("Correct Answer: ")[-1].strip()
                # Case-insensitive comparison after stripping whitespace
                is_correct = gpt_answer.strip().upper() == correct_ans.strip().upper()
            else:
                gpt_answer = "Error: No answer in correct format"
                is_correct = "N/A"
        
        responses_data.append([gpt_answer, time_end-time_start, is_correct])
        print(f"\nAnswer: {gpt_answer}")
        print(f"Time taken: {time_end-time_start:.2f} seconds")
        print(f"Correct? {is_correct}")
        
        # Increase delay between requests to 15 seconds
        time.sleep(max(15 - (time_end-time_start), 0))
        
    except Exception as e:
        print(f"Error processing question {i}:", str(e))
        responses_data.append([f"Error: {str(e)}", 0, "N/A"])

# Create Results directory if it doesn't exist
results_dir = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Results/1_Claude 3.7_Final Results with Explanation/"
os.makedirs(results_dir, exist_ok=True)

results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
results_df.to_csv(results_dir + "VLAT_" + str(int(time.time())) + ".csv", index_label="id")
print("\n*** Finished ***")


Processing question 1:
{'id': 1, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_1', 'question: ': 'What was the price of a barrel of oil in February 2015? ', 'option:': '$39.72; $57.36; $48.90; $62.85', 'correct': '$48.90 '}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: What was the price of a barrel of oil in February 2015? 
Options: $39.72; $57.36; $48.90; $62.85
Correct answer: $48.90

API Response: {'id': 'msg_01QJiSqfL49gJw15BBmMqmMG', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': "## Task 1: Data Extraction and Table Creation\n\nNumerical values on the y-axis (Oil Price in $ per barrel):\n- 35\n- 40\n- 45\n- 50\n- 55\n- 60\n- 65\n\nMonths on the x-axis:\n- January\n- February\n- March\n- April\n- May\n- June\n- July\n- August\n- September\n- October\n- November\n- December\n\nData points from the graph:\n\n| Month | Oil Price ($ per barrel) |\n|-----


Processing question 4:
{'id': 4, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_4', 'question: ': 'Over the course of the second half of 2015, the price of a barrel of oil was ____________.', 'option:': 'rising; falling; staying', 'correct': 'rising'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: Over the course of the second half of 2015, the price of a barrel of oil was ____________.
Options: rising; falling; staying
Correct answer: rising

API Response: {'id': 'msg_01VecjkitbKHtmfZxPGFkcTJ', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': "# Task 1: Data Extraction and Table Creation\n\nLooking at the graph, I can identify the following values on the axes:\n\nY-axis (Oil Price in $ per barrel): Values range from approximately 35 to 65 dollars per barrel\nX-axis (Month): January through December 2015\n\nHere are the specific data points I can identify from


Processing question 7:
{'id': 7, 'dropped': 'no', 'vis': 'VLAT_b', 'item': 'b_2', 'question: ': 'In which country is the average internet speed the fastest in Asia?', 'option:': 'Japan; South Korea; Sri Lanka; India', 'correct': 'Sri Lanka'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_b.png
Question: In which country is the average internet speed the fastest in Asia?
Options: Japan; South Korea; Sri Lanka; India
Correct answer: Sri Lanka

API Response: {'id': 'msg_018x6bfnaSKVkNZPjdPbrp3s', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': '# Task 1: Data Extraction and Table Creation\n\nI can identify the following values from the x-axis (countries) and y-axis (speed in Mbps):\n\n| Country | Average Internet Speed (Mbps) |\n|---------|------------------------------|\n| Australia | 14 |\n| China | 15 |\n| Hong Kong | 3 |\n| India | 10 |\n| Indonesia | 18 |\n| Japan | 17 |


Processing question 10:
{'id': 10, 'dropped': 'no', 'vis': 'VLAT_c', 'item': 'c_1', 'question: ': 'What is the cost of peanuts in Las Vegas?', 'option:': '$5; $15; $23.4; $35.4', 'correct': '$5 '}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_c.png
Question: What is the cost of peanuts in Las Vegas?
Options: $5; $15; $23.4; $35.4
Correct answer: $5

API Response: {'id': 'msg_014RHbxTrfRA7e2JkAozMrpF', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': '# Task 1: Data Extraction and Table Creation\n\nLooking at the graph "Hotel Costs of Room Service" which shows costs in dollars for different items across various cities.\n\nThe y-axis shows Cost ($) from 0 to 80 dollars.\nThe x-axis shows different cities: New York City, Las Vegas, Honolulu, Atlanta, Boston, Washington D.C., Chicago, Orlando, San Francisco, and Seattle.\n\nThe graph is a stacked bar chart showing costs for:\n


API Response: {'id': 'msg_013SfMXcWtgC5buL42eN8bAn', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': '# Task 1: Data Extraction and Table Creation\n\nLooking at the graph "Hotel Costs of Room Service" which shows costs in different cities broken down by item categories (Vodka, Soda, Peanuts, Water, Sandwich).\n\nThe y-axis shows Cost ($) from 0 to 80 dollars.\nThe x-axis shows different cities: New York, Las Vegas, Honolulu, Atlanta, Boston, Washington D.C., Chicago, Orlando, San Francisco, and Seattle.\n\nLet me create a table with the approximate values for each item in each city:\n\n| City | Sandwich | Water | Peanuts | Soda | Vodka | Total Cost ($) |\n|------|----------|-------|---------|------|-------|---------------|\n| New York | ~12 | ~8 | ~7 | ~9 | ~19 | ~55 |\n| Las Vegas | ~10 | ~7 | ~5 | ~8 | ~15 | ~45 |\n| Honolulu | ~15 | ~7 | ~6 | ~8 | ~28 | ~64 |\n| Atlanta | ~10 | ~3 | ~5 | ~15 | ~0 | ~33 |\n| Boston 

...


API Response: {'id': 'msg_01HVqeqqEQi36YLDHsy5kTQA', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': '# Task 1: Data Extraction and Table Creation\n\nLooking at the graph, I can identify the following numerical values:\n\nX-axis (Number of Stations): Values range from approximately 100 to 450\nY-axis (Total System Length): Values range from approximately 150 to 600 km\nBubble size: Represents ridership (bn per year), with legend showing 1.5, 2.5, and 3.5 billion\n\nHere\'s a structured table with all data points:\n\n| City | Number of Stations | Total System Length (km) | Ridership (bn per year) |\n|------|---------------------|--------------------------|-------------------------|\n| City A | 225 | 250 | ~2.5 |\n| City B | 425 | 525 | ~3.5 |\n| City C | 350 | 325 | ~2.5 |\n| City D | 200 | 360 | ~1.5 |\n| City E | 275 | 375 | ~2.5 |\n| City F | 425 | 350 | ~2.0 |\n| City G | 150 | 475 | ~1.0 |\n| City H | 175 | 290 | 


Processing question 55:
{'id': 55, 'dropped': 'no', 'vis': 'VLAT_k', 'item': 'k_1', 'question: ': 'What was the unemployment rate for Indiana (IN) in 2015?', 'option:': '1.1% - 2.3%; 2.3% - 3.4%; 3.4% - 4.6%; 4.6% - 5.7%', 'correct': '1.1% - 2.3%'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_k.png
Question: What was the unemployment rate for Indiana (IN) in 2015?
Options: 1.1% - 2.3%; 2.3% - 3.4%; 3.4% - 4.6%; 4.6% - 5.7%
Correct answer: 1.1% - 2.3%

API Response: {'id': 'msg_01A42hw8nKML9J87jtefrKuL', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': "# Task 1: Data Extraction and Table Creation\n\nLooking at the map, I can identify the following ranges from the legend:\n- 0.0% - 1.1%\n- 1.1% - 2.3%\n- 2.3% - 3.4%\n- 3.4% - 4.6%\n- 4.6% - 5.8%\n- 5.8% - 6.9%\n- 6.9% and above\n\nNow, I'll create a table with all states and their unemployment rate ranges in 2015:\n\n| Sta


Processing question 57:
{'id': 57, 'dropped': 'no', 'vis': 'VLAT_k', 'item': 'k_3', 'question: ': 'In 2015, the unemployment rate for Washington (WA) was higher than that of Arizona (AZ).', 'option:': 'True; False', 'correct': 'FALSE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_k.png
Question: In 2015, the unemployment rate for Washington (WA) was higher than that of Arizona (AZ).
Options: True; False
Correct answer: FALSE

API Response: {'id': 'msg_01VzHzYA7yncqtkv5Moi8eHg', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': "# Task 1: Data Extraction and Table Creation\n\nLooking at the map, I can identify the following ranges from the legend:\n- 0.0% to 1.1%\n- 1.1% to 2.3%\n- 2.3% to 3.4%\n- 3.4% to 4.6%\n- 4.6% to 5.8%\n- 5.8% to 6.9%\n\nBased on the color coding in the map, I can identify the unemployment rates for each state. Specifically focusing on Washington (WA


Processing question 59:
{'id': 59, 'dropped': 'no', 'vis': 'VLAT_l', 'item': 'l_2', 'question: ': 'For which website was the number of unique visitors the largest in 2010?', 'option:': 'Facebook; Amazon; Bing; Google', 'correct': 'Google'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_l.png
Question: For which website was the number of unique visitors the largest in 2010?
Options: Facebook; Amazon; Bing; Google
Correct answer: Google

API Response: {'id': 'msg_016ocXTJ2Rk8gYamj3ofhTgJ', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': "I need to analyze this image carefully to answer your question about unique visitors in 2010.\n\nLooking at the image, I can see this is a treemap visualization showing different websites categorized by sector (Retail, Financial, Social Media, Search, News, Computer). The size of each rectangle appears to represent some metric, which in this


*** Finished ***
